# EE542 Lab 6 · Hands-on CUDA: from a triple loop to Tensor Cores

This notebook is not "click Run". **Every program is written out in full in the cells**, with an explanation and a small experiment for each one. Suggested rhythm:

1. Read the explanation
2. Read the code and **predict** the result
3. Run it and compare with your prediction
4. Do the **Try it** experiment

In Colab: **Runtime → Change runtime type → T4 GPU**, then run top to bottom.

> Sections 0–7 build every matrix-multiply kernel from scratch. Section 8 runs the full graded benchmark from the repository, Section 9 is the Python shared library (Part 7), Section 10 is image convolution (Part 8).

## 0. Meet your GPU

Before writing CUDA, find out what hardware you have. This small program prints the numbers that matter:

| Property | Why it matters |
|---|---|
| Number of SMs | A GPU is dozens of *streaming multiprocessors*; each runs several thread blocks at once. More SMs = more parallelism |
| Warp size = 32 | The hardware executes **32 threads together, on the same instruction**. This is the unit to think in for performance |
| Shared memory / block | Fast on-chip memory a thread block can use; tiling (Section 5) is built on it |
| Memory bandwidth | How fast data moves from DRAM to the chip; it is the ceiling for many kernels |

In [ ]:
%%writefile device_query.cu
#include <cstdio>

int main()
{
    cudaDeviceProp p;
    cudaGetDeviceProperties(&p, 0);
    printf("GPU                   : %s\n", p.name);
    printf("Compute capability    : %d.%d\n", p.major, p.minor);
    printf("Streaming MPs (SMs)   : %d\n", p.multiProcessorCount);
    printf("Warp size             : %d threads\n", p.warpSize);
    printf("Max threads / block   : %d\n", p.maxThreadsPerBlock);
    printf("Shared memory / block : %zu KB\n", p.sharedMemPerBlock / 1024);
    printf("Registers / block     : %d\n", p.regsPerBlock);
    printf("Global memory         : %.1f GB\n", p.totalGlobalMem / 1e9);
    int memClockKHz = 0, busBits = 0;
    cudaDeviceGetAttribute(&memClockKHz, cudaDevAttrMemoryClockRate, 0);
    cudaDeviceGetAttribute(&busBits, cudaDevAttrGlobalMemoryBusWidth, 0);
    // DDR memory transfers twice per clock.
    printf("Peak memory bandwidth : %.0f GB/s\n", 2.0 * memClockKHz * 1e3 * (busBits / 8) / 1e9);
    return 0;
}

In [ ]:
!nvcc -O3 -arch=native -o device_query device_query.cu && ./device_query

**Compare with the T4 datasheet**: 40 SMs, FP32 peak ≈ **8.1 TFLOP/s**, FP16 Tensor Core ≈ **65 TFLOP/s**, memory bandwidth ≈ **320 GB/s**.

Remember two numbers: **8100 GFLOP/s** (compute ceiling) and **320 GB/s** (bandwidth ceiling). For every kernel below, ask: which ceiling is it closer to?

## 1. CPU baseline: same arithmetic, different speed

Multiplying C = A × B (N×N) takes N³ multiply-adds = **2N³ floating-point operations**. Dividing that by the time gives **GFLOP/s**, which lets us compare different sizes, and CPU against GPU, on one scale.

The two functions below do **exactly the same arithmetic**; only the loop order differs:

- `mm_ijk` (the handout's version): the inner loop runs over k, so `B[k*N + j]` jumps N floats every iteration, **a cache miss almost every time**
- `mm_ikj`: the inner loop runs over j, so A, B and C are all read **contiguously**; the cache works and the compiler can vectorise with SIMD

**Predict**: how many times faster is `ikj` than `ijk`?

In [ ]:
%%writefile cpu_mm.c
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

static double now_s(void)
{
    struct timespec t;
    clock_gettime(CLOCK_MONOTONIC, &t);   // wall clock, not clock() (CPU time)
    return t.tv_sec + t.tv_nsec * 1e-9;
}

// Handout order: the inner loop walks DOWN a column of B (stride N floats).
void mm_ijk(const float *A, const float *B, float *C, int N)
{
    for (int i = 0; i < N; i++)
        for (int j = 0; j < N; j++) {
            float sum = 0.0f;
            for (int k = 0; k < N; k++)
                sum += A[i * N + k] * B[k * N + j];
            C[i * N + j] = sum;
        }
}

// Same arithmetic, inner loop walks ALONG rows of B and C (stride 1).
void mm_ikj(const float *A, const float *B, float *C, int N)
{
    memset(C, 0, (size_t)N * N * sizeof(float));
    for (int i = 0; i < N; i++)
        for (int k = 0; k < N; k++) {
            float a = A[i * N + k];
            for (int j = 0; j < N; j++)
                C[i * N + j] += a * B[k * N + j];
        }
}

int main(int argc, char **argv)
{
    int N = argc > 1 ? atoi(argv[1]) : 512;
    size_t bytes = (size_t)N * N * sizeof(float);
    float *A = malloc(bytes), *B = malloc(bytes), *C1 = malloc(bytes), *C2 = malloc(bytes);
    for (int i = 0; i < N * N; i++) {
        A[i] = rand() / (float)RAND_MAX;
        B[i] = rand() / (float)RAND_MAX;
    }
    double t0 = now_s(); mm_ijk(A, B, C1, N); double t_ijk = now_s() - t0;
    t0 = now_s();        mm_ikj(A, B, C2, N); double t_ikj = now_s() - t0;

    double maxdiff = 0;
    for (int i = 0; i < N * N; i++) {
        double d = C1[i] - C2[i];
        if (d < 0) d = -d;
        if (d > maxdiff) maxdiff = d;
    }
    double flops = 2.0 * N * N * (double)N;
    printf("N=%d  ijk: %8.1f ms  %6.2f GFLOP/s\n", N, t_ijk * 1e3, flops / t_ijk / 1e9);
    printf("N=%d  ikj: %8.1f ms  %6.2f GFLOP/s   (%.1fx faster, max diff %.1e)\n",
           N, t_ikj * 1e3, flops / t_ikj / 1e9, t_ijk / t_ikj, maxdiff);
    return 0;
}

In [ ]:
!gcc -O3 -march=native -o cpu_mm cpu_mm.c
!for N in 256 512 1024; do ./cpu_mm $N; done

**Observation**: identical result, an order of magnitude faster just by reordering loops. That is the power of the **memory access pattern**, and "coalescing" on the GPU (Section 4) is the same idea.

**Takeaway**: compare the GPU against **optimised CPU code**. Otherwise "the GPU is 1000× faster" only means the CPU code was bad.

**Try it**: replace `-O3 -march=native` with `-O0` and run again. Which version suffers more, and why? (Hint: the compiler can vectorise the inner loop of `ikj`.)

## 2. Your first CUDA program: how threads are numbered

The CUDA execution model:

```
grid  = many blocks
block = many threads; threads in one block can share shared memory and synchronise
```

Each thread uses `blockIdx` (which block am I in), `blockDim` (threads per block) and `threadIdx` (my index inside the block) to work out **which data it is responsible for**:

```
global index = blockIdx.x * blockDim.x + threadIdx.x
```

Below we launch 2 blocks × 4 threads and let each thread print its index.

**Predict**: is the output in the order 0, 1, 2, …, 7?

In [ ]:
%%writefile hello_threads.cu
#include <cstdio>

__global__ void hello()
{
    int global = blockIdx.x * blockDim.x + threadIdx.x;
    printf("block %d, thread %d  ->  global index %d\n", blockIdx.x, threadIdx.x, global);
}

int main()
{
    hello<<<2, 4>>>();          // <<<number of blocks, threads per block>>>
    cudaDeviceSynchronize();    // kernels are asynchronous: wait, or main() exits first
    return 0;
}

In [ ]:
!nvcc -arch=native -o hello_threads hello_threads.cu && ./hello_threads

**Observations**:
- Threads of one block usually appear in order (a warp executes together), but **the order between blocks is not guaranteed**. Never rely on thread execution order.
- Remove `cudaDeviceSynchronize()` and try again: maybe nothing is printed, because kernel launches are **asynchronous** and the CPU does not wait.

Matrices are 2-D, so matrix multiply uses 2-D blocks and grids:

```
row = blockIdx.y * blockDim.y + threadIdx.y
col = blockIdx.x * blockDim.x + threadIdx.x
```

## 3. Test harness: allocate, copy, time, verify

Every kernel needs the same chores: allocate GPU memory, copy data, time it, copy back, check the answer. They live in one header, `mm_harness.cuh`, so **each kernel file only contains the kernel and a `launch()` function**.

Read this file line by line once: it is the skeleton of every CUDA program.

1. `cudaMalloc`: allocate **device memory** (the GPU cannot read your CPU's memory directly)
2. `cudaMemcpy(..., HostToDevice)`: move data to the GPU over PCIe
3. One **warm-up** launch first (the first launch has extra overhead), then average many runs
4. Time with **`cudaEvent`**: timestamps recorded on the GPU, more accurate than a CPU timer
5. Copy the result back, pick 64 random entries, recompute them on the CPU in double precision and compare

In [ ]:
%%writefile mm_harness.cuh
#pragma once
#include <cstdio>
#include <cstdlib>
#include <cmath>
#include <vector>

#define CHECK(x) do { cudaError_t e = (x); if (e != cudaSuccess) {                 \
    printf("CUDA error: %s at %s:%d\n", cudaGetErrorString(e), __FILE__, __LINE__); \
    exit(1); } } while (0)

// Each kernel file implements this: launch the kernel on device pointers.
void launch(const float *dA, const float *dB, float *dC, int N);

int run(int argc, char **argv, const char *name)
{
    int N = argc > 1 ? atoi(argv[1]) : 1024;
    int reps = argc > 2 ? atoi(argv[2]) : 10;
    size_t bytes = (size_t)N * N * sizeof(float);

    // 1. Host data
    std::vector<float> A(N * N), B(N * N), C(N * N);
    srand(42);
    for (int i = 0; i < N * N; i++) {
        A[i] = rand() / (float)RAND_MAX * 2 - 1;
        B[i] = rand() / (float)RAND_MAX * 2 - 1;
    }

    // 2. Device memory and host -> device copies
    float *dA, *dB, *dC;
    CHECK(cudaMalloc(&dA, bytes));
    CHECK(cudaMalloc(&dB, bytes));
    CHECK(cudaMalloc(&dC, bytes));
    CHECK(cudaMemcpy(dA, A.data(), bytes, cudaMemcpyHostToDevice));
    CHECK(cudaMemcpy(dB, B.data(), bytes, cudaMemcpyHostToDevice));

    // 3. Warm-up, then timed repetitions
    launch(dA, dB, dC, N);
    CHECK(cudaGetLastError());          // catches bad launch configurations
    CHECK(cudaDeviceSynchronize());     // catches errors inside the kernel
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    cudaEventRecord(start);
    for (int r = 0; r < reps; r++) launch(dA, dB, dC, N);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    float total_ms;
    cudaEventElapsedTime(&total_ms, start, stop);
    double ms = total_ms / reps;

    // 4. Copy back and spot-check 64 random entries against a double-precision dot product
    CHECK(cudaMemcpy(C.data(), dC, bytes, cudaMemcpyDeviceToHost));
    double maxerr = 0;
    for (int t = 0; t < 64; t++) {
        int i = rand() % N, j = rand() % N;
        double ref = 0;
        for (int k = 0; k < N; k++) ref += (double)A[i * N + k] * B[k * N + j];
        maxerr = fmax(maxerr, fabs(ref - C[i * N + j]));
    }

    printf("%-12s N=%5d  %9.3f ms  %8.1f GFLOP/s  max err %.1e %s\n", name, N, ms,
           2.0 * N * N * (double)N / (ms * 1e-3) / 1e9, maxerr, maxerr < 1e-2 ? "OK" : "WRONG!");
    cudaFree(dA); cudaFree(dB); cudaFree(dC);
    return 0;
}

## 4. Naive kernel: one thread per output

The most direct version (the handout's): launch N×N threads, **each computes one element of C**, the dot product of a row of A and a column of B.

```
thread (row, col)  →  C[row][col] = Σ_k  A[row][k] · B[k][col]
```

Each thread reads 2N floats and does 2N operations: **one operation per 4 bytes read**. If every read really went to DRAM at 320 GB/s, the ceiling would be about 80 GFLOP/s. The L1/L2 caches help, but this is still far from 8100 GFLOP/s.

**Predict**: roughly how many GFLOP/s at N = 2048?

In [ ]:
%%writefile mm_naive.cu
#include "mm_harness.cuh"

__global__ void mm_naive(const float *A, const float *B, float *C, int N)
{
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;   // x -> column: neighbours in a warp share a row
    if (row < N && col < N) {                           // the grid may overhang the matrix
        float sum = 0.0f;
        for (int k = 0; k < N; k++)
            sum += A[row * N + k] * B[k * N + col];
        C[row * N + col] = sum;
    }
}

void launch(const float *dA, const float *dB, float *dC, int N)
{
    dim3 block(16, 16);                                  // 256 threads per block
    dim3 grid((N + 15) / 16, (N + 15) / 16);             // round up so every element is covered
    mm_naive<<<grid, block>>>(dA, dB, dC, N);
}

int main(int argc, char **argv) { return run(argc, argv, "naive"); }

In [ ]:
!nvcc -O3 -arch=native -o mm_naive mm_naive.cu
!for N in 512 1024 2048; do ./mm_naive $N; done

### Experiment: memory coalescing

GPU memory is accessed **per warp (32 threads)**. If the 32 threads read **32 consecutive floats**, the hardware serves them in one transaction ("coalesced"); if the addresses are scattered, it needs many.

In the kernel above `threadIdx.x` maps to **col**, so the 32 threads of a warp handle 32 neighbouring columns of the same row:
- reading `B[k*N + col]`: 32 consecutive addresses → **coalesced** ✓
- reading `A[row*N + k]`: all 32 threads read the same address → **broadcast**, also fast ✓

The version below changes one thing only: **x and y are swapped**, so `threadIdx.x` maps to row. It computes exactly the same thing.

**Predict**: how much slower is it?

In [ ]:
%%writefile mm_naive_swapped.cu
#include "mm_harness.cuh"

// Identical arithmetic; only the thread -> (row, col) mapping is swapped.
__global__ void mm_naive_swapped(const float *A, const float *B, float *C, int N)
{
    int row = blockIdx.x * blockDim.x + threadIdx.x;   // x -> row: neighbours in a warp are N floats apart
    int col = blockIdx.y * blockDim.y + threadIdx.y;
    if (row < N && col < N) {
        float sum = 0.0f;
        for (int k = 0; k < N; k++)
            sum += A[row * N + k] * B[k * N + col];
        C[row * N + col] = sum;
    }
}

void launch(const float *dA, const float *dB, float *dC, int N)
{
    dim3 block(16, 16), grid((N + 15) / 16, (N + 15) / 16);
    mm_naive_swapped<<<grid, block>>>(dA, dB, dC, N);
}

int main(int argc, char **argv) { return run(argc, argv, "swapped"); }

In [ ]:
!nvcc -O3 -arch=native -o mm_naive_swapped mm_naive_swapped.cu
!./mm_naive 2048; ./mm_naive_swapped 2048

**Observation**: same result, several times slower. Now neighbouring threads in a warp read A at addresses N floats apart, and write C the same way, so every access splits into many memory transactions.

**Rule**: make `threadIdx.x` (neighbouring threads in a warp) access **neighbouring addresses**. This is the first rule of CUDA optimisation.

## 5. Shared-memory tiling: data reuse

The naive kernel's problem: **every value is re-read from DRAM**. The same row of A is read separately by each of the N threads computing that row of C.

**The tiling idea**: one block owns a 16×16 tile of C. It needs 16 rows of A and 16 columns of B, processed in chunks of 16:

```
for each chunk m:
    the whole block copies one 16×16 tile of A and one of B into shared memory   ← read DRAM once
    __syncthreads()                                                             ← wait until all copies are done
    each thread does 16 multiply-adds from shared memory                        ← every value used 16 times
    __syncthreads()                                                             ← wait until everyone is done before overwriting
```

DRAM traffic drops to **1/16**. Shared memory is on the chip, more than an order of magnitude faster than DRAM.

**Both `__syncthreads()` calls are required**; the experiment below proves it.

In [ ]:
%%writefile mm_tiled.cu
#include "mm_harness.cuh"
#ifndef TILE
#define TILE 16
#endif

__global__ void mm_tiled(const float *A, const float *B, float *C, int N)
{
    __shared__ float As[TILE][TILE];      // on-chip, shared by the whole block
    __shared__ float Bs[TILE][TILE];
    int tx = threadIdx.x, ty = threadIdx.y;
    int row = blockIdx.y * TILE + ty;
    int col = blockIdx.x * TILE + tx;
    float acc = 0.0f;

    for (int m = 0; m < (N + TILE - 1) / TILE; ++m) {
        // Each thread loads ONE element of each tile (zero outside the matrix).
        int aCol = m * TILE + tx, bRow = m * TILE + ty;
        As[ty][tx] = (row < N && aCol < N) ? A[row * N + aCol] : 0.0f;
        Bs[ty][tx] = (bRow < N && col < N) ? B[bRow * N + col] : 0.0f;
        __syncthreads();                  // (1) tiles complete before anyone reads them

        for (int k = 0; k < TILE; ++k)
            acc += As[ty][k] * Bs[k][tx];
#ifndef SKIP_SECOND_SYNC
        __syncthreads();                  // (2) everyone finished before the tiles are overwritten
#endif
    }
    if (row < N && col < N)
        C[row * N + col] = acc;
}

void launch(const float *dA, const float *dB, float *dC, int N)
{
    dim3 block(TILE, TILE), grid((N + TILE - 1) / TILE, (N + TILE - 1) / TILE);
    mm_tiled<<<grid, block>>>(dA, dB, dC, N);
}

int main(int argc, char **argv) { return run(argc, argv, "tiled"); }

In [ ]:
!nvcc -O3 -arch=native -o mm_tiled16 mm_tiled.cu -DTILE=16
!nvcc -O3 -arch=native -o mm_tiled32 mm_tiled.cu -DTILE=32
!for N in 512 1024 2048; do ./mm_naive $N; ./mm_tiled16 $N; ./mm_tiled32 $N; done

### Experiment: remove the second `__syncthreads()`

Compiling with `-DSKIP_SECOND_SYNC` drops the second barrier. Fast threads move on to the next chunk and **start overwriting the shared tiles while slow threads are still computing with the old data**. This is a **race condition**.

**Predict**: will the result be wrong? Every time?

In [ ]:
!nvcc -O3 -arch=native -o mm_tiled_race mm_tiled.cu -DSKIP_SECOND_SYNC
!for i in 1 2 3; do ./mm_tiled_race 1024; done

**Observation**: you will most likely see `WRONG!`, and the error may differ from run to run. That is what makes races dangerous: they are **non-deterministic**; sometimes the answer is right by luck, so tests can miss them.

### Why is tiling only a little faster?

Look at the inner loop: `acc += As[ty][k] * Bs[k][tx]`. Each **multiply-add** needs **2 reads** from shared memory. The bottleneck moved from DRAM to shared memory. To go faster, every value we load must take part in more arithmetic. That is the next section.

## 6. Extension 1: register blocking

**Core idea**: each thread computes not 1 output but **8×8 = 64**, keeping them in its own **registers** (the fastest storage there is).

```
one block (256 threads) owns a 128×128 tile of C
    = 16×16 threads, each owning an 8×8 sub-tile

inner step d:
    read 8 values of A (regM) and 8 values of B (regN) from shared memory   ← 16 reads
    do 8×8 = 64 multiply-adds (an outer product)                           ← 64 FMAs
```

That is **4 multiply-adds per shared-memory read**, versus 0.5 in the tiled kernel. This step is usually what takes you from "a small fraction of cuBLAS" to "most of cuBLAS".

**The cost**: each thread needs 64+ registers, so fewer threads fit on an SM (**lower occupancy**). But each thread has so much independent work that it hides the latency anyway.

To keep the code readable, this kernel requires N to be a multiple of 128 (no boundary handling).

In [ ]:
%%writefile mm_regblock.cu
#include "mm_harness.cuh"

// Block tile BM x BN of C, K-step BK, each thread computes TM x TN outputs.
#define BM 128
#define BN 128
#define BK 8
#define TM 8
#define TN 8
#define NTHREADS ((BM * BN) / (TM * TN))      // 256

__global__ void __launch_bounds__(NTHREADS) mm_regblock(const float *A, const float *B, float *C, int N)
{
    __shared__ float As[BM * BK];             // 128 x 8 slice of A
    __shared__ float Bs[BK * BN];             //   8 x 128 slice of B

    // This thread's 8x8 sub-tile inside the block's 128x128 tile.
    const int threadCol = threadIdx.x % (BN / TN);   // 0..15
    const int threadRow = threadIdx.x / (BN / TN);   // 0..15

    // Move the pointers to this block's tiles.
    A += blockIdx.y * BM * N;
    B += blockIdx.x * BN;
    C += blockIdx.y * BM * N + blockIdx.x * BN;

    // Loading pattern: 256 threads fill 128x8 (A) and 8x128 (B) floats, 4 each.
    const int aRow = threadIdx.x / BK, aCol = threadIdx.x % BK;   // 32 rows per pass
    const int bRow = threadIdx.x / BN, bCol = threadIdx.x % BN;   //  2 rows per pass

    float acc[TM * TN] = {0.0f};              // 64 results in registers
    float regM[TM], regN[TN];

    for (int bk = 0; bk < N; bk += BK) {
        for (int o = 0; o < BM; o += NTHREADS / BK) As[(aRow + o) * BK + aCol] = A[(aRow + o) * N + aCol];
        for (int o = 0; o < BK; o += NTHREADS / BN) Bs[(bRow + o) * BN + bCol] = B[(bRow + o) * N + bCol];
        __syncthreads();
        A += BK;
        B += BK * N;

        for (int d = 0; d < BK; ++d) {
            for (int i = 0; i < TM; ++i) regM[i] = As[(threadRow * TM + i) * BK + d];   // 8 values of A
            for (int i = 0; i < TN; ++i) regN[i] = Bs[d * BN + threadCol * TN + i];     // 8 values of B
            for (int m = 0; m < TM; ++m)                                                 // 64 FMAs
                for (int n = 0; n < TN; ++n)
                    acc[m * TN + n] += regM[m] * regN[n];
        }
        __syncthreads();
    }
    for (int m = 0; m < TM; ++m)
        for (int n = 0; n < TN; ++n)
            C[(threadRow * TM + m) * N + threadCol * TN + n] = acc[m * TN + n];
}

void launch(const float *dA, const float *dB, float *dC, int N)
{
    if (N % BM) { printf("N must be a multiple of %d\n", BM); exit(1); }
    mm_regblock<<<dim3(N / BN, N / BM), NTHREADS>>>(dA, dB, dC, N);
}

int main(int argc, char **argv) { return run(argc, argv, "regblock"); }

In [ ]:
!nvcc -O3 -arch=native -o mm_regblock mm_regblock.cu
!for N in 1024 2048 4096; do ./mm_tiled16 $N; ./mm_regblock $N; done

**Try it**: change `TM` and `TN` to 4 (each thread computes 4×4) and rebuild. `NTHREADS` becomes 1024 automatically. Which is faster, and why? (Hint: arithmetic per load, and how many threads fit on an SM.)

**Register usage**: the line below asks the compiler how many registers each kernel uses, and whether anything spills to memory.

In [ ]:
!nvcc -O3 -arch=native -Xptxas -v -c mm_regblock.cu -o /dev/null 2>&1 | grep -E "registers|spill"

## 7. cuBLAS and Tensor Cores

### cuBLAS: NVIDIA's library, our ceiling

One trap: **cuBLAS stores matrices column-major** (the Fortran convention), while we use row-major. The trick:

```
row-major C = A × B   ⟺   column-major Cᵀ = Bᵀ × Aᵀ
```

A row-major buffer read as column-major *is* its transpose. So we **pass B before A** and get a row-major C directly, without transposing anything.

In [ ]:
%%writefile mm_cublas.cu
#include "mm_harness.cuh"
#include <cublas_v2.h>

void launch(const float *dA, const float *dB, float *dC, int N)
{
    static cublasHandle_t handle = nullptr;
    if (!handle) cublasCreate(&handle);       // create once, reuse
    const float alpha = 1.0f, beta = 0.0f;    // C = alpha * op(A) op(B) + beta * C
    // Row-major C = A * B  ==  column-major C^T = B^T * A^T  ->  pass B first.
    cublasSgemm(handle, CUBLAS_OP_N, CUBLAS_OP_N, N, N, N,
                &alpha, dB, N, dA, N, &beta, dC, N);
}

int main(int argc, char **argv) { return run(argc, argv, "cublas"); }

In [ ]:
!nvcc -O3 -arch=native -o mm_cublas mm_cublas.cu -lcublas
!for N in 1024 2048 4096; do ./mm_regblock $N; ./mm_cublas $N; done

**Compute it**: what percentage of cuBLAS is your `regblock`? That percentage is the most convincing number for a résumé.

Why is cuBLAS still faster? It is hand-tuned for every GPU generation: double buffering (load the next tile while computing this one), warp-level tiling, a kernel picked per matrix shape, even instruction scheduling at the assembly level.

### Tensor Cores: dedicated matrix-multiply hardware

The T4 has **Tensor Cores**: one instruction performs a whole 16×16×16 matrix multiply, but the inputs must be **FP16 (half precision)**. CUDA's WMMA API uses them per warp: each warp owns one 16×16 tile of C.

**Predict**: how much larger will the error be with FP16 inputs? (FP32 has about 7 significant digits, FP16 about 3.)

In [ ]:
%%writefile mm_wmma.cu
#include "mm_harness.cuh"
#include <cuda_fp16.h>
#include <mma.h>
using namespace nvcuda;

// One warp computes one 16x16 tile of C. Inputs FP16, accumulation FP32.
__global__ void mm_wmma(const half *A, const half *B, float *C, int N)
{
    int tileRow = (blockIdx.x * blockDim.x + threadIdx.x) / 32;   // which warp am I (along M)
    int tileCol = blockIdx.y * blockDim.y + threadIdx.y;          // along N
    if (tileRow * 16 >= N || tileCol * 16 >= N) return;           // whole warp exits together

    wmma::fragment<wmma::matrix_a, 16, 16, 16, half, wmma::row_major> a;
    wmma::fragment<wmma::matrix_b, 16, 16, 16, half, wmma::row_major> b;
    wmma::fragment<wmma::accumulator, 16, 16, 16, float> c;
    wmma::fill_fragment(c, 0.0f);
    for (int k = 0; k < N; k += 16) {
        wmma::load_matrix_sync(a, A + tileRow * 16 * N + k, N);
        wmma::load_matrix_sync(b, B + k * N + tileCol * 16, N);
        wmma::mma_sync(c, a, b, c);                               // the Tensor Core instruction
    }
    wmma::store_matrix_sync(C + tileRow * 16 * N + tileCol * 16, c, N, wmma::mem_row_major);
}

__global__ void to_half(const float *in, half *out, int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) out[i] = __float2half(in[i]);
}

void launch(const float *dA, const float *dB, float *dC, int N)
{
    static half *hA = nullptr, *hB = nullptr;   // FP16 copies, converted once (warm-up call)
    if (!hA) {
        cudaMalloc(&hA, (size_t)N * N * sizeof(half));
        cudaMalloc(&hB, (size_t)N * N * sizeof(half));
        to_half<<<(N * N + 255) / 256, 256>>>(dA, hA, N * N);
        to_half<<<(N * N + 255) / 256, 256>>>(dB, hB, N * N);
    }
    dim3 block(128, 4);                         // 4 x 4 = 16 warps per block
    dim3 grid((N / 16 + 3) / 4, (N / 16 + 3) / 4);
    mm_wmma<<<grid, block>>>(hA, hB, dC, N);
}

int main(int argc, char **argv) { return run(argc, argv, "wmma_fp16"); }

In [ ]:
!nvcc -O3 -arch=native -o mm_wmma mm_wmma.cu
!for N in 1024 2048 4096; do ./mm_cublas $N; ./mm_wmma $N; done

**Observations**:
- Speed: even this simplest WMMA kernel (reading straight from DRAM, no shared memory) can approach or beat FP32 cuBLAS
- Error: `max err` is clearly larger. **Speed is not free**: deep learning tolerates FP16 error, scientific computing often does not

> Note: the harness threshold is 1e-2, so FP16 may print `WRONG!` at large N. That is lost precision, not a bug. Think about it: why does the error grow with N?

## 8. The full benchmark (graded results)

Sections 4–7 rebuilt each kernel by hand. The repository has all of them in one program, `bin/sgemm`, plus the CPU baselines, and a script that sweeps N and plots the results. This is what goes into the report.

Note: these cells use `!cd lab6 && ...` instead of `%cd`, so the working directory of the later sections does not change.

In [ ]:
!git clone -q https://github.com/Tr3nt-Xie/ee542-lab6.git lab6 2>/dev/null || git -C lab6 pull -q
!cd lab6 && make -j4 all 2>&1 | tail -3

One size, every implementation. Columns: `kernel_ms` (GPU time only), `gflops`, `e2e_ms` (including PCIe copies both ways), `max_rel_err` (against cuBLAS).

In [ ]:
!cd lab6 && bin/sgemm all 2048 10 | column -t -s,

Full sweep, N = 256 … 4096 (takes a few minutes; the naive CPU loop is the slow part, so it stops at N = 2048), then the plot.

In [ ]:
!cd lab6 && bash scripts/run_all.sh > /dev/null && python3 scripts/plot.py
from IPython.display import Image as Img, display
display(Img("lab6/results/results.png"))

**Read the plot**:
- Left (log-log time vs N): every line has slope ≈ 3, because the work is 2N³. The vertical gaps between lines are the speed-ups
- Right: GFLOP/s at the largest N, as a percentage of cuBLAS FP32

**Kernel time vs end-to-end time**: compare `kernel_ms` and `e2e_ms` in the table. Copying 3 matrices of N² floats over PCIe (≈ 12 GB/s on Colab) often costs **more than the multiply itself** for the fast kernels. In real applications, keep data on the GPU across many operations.

### Optional: profile with Nsight Compute

`ncu` shows what the hardware is actually doing: DRAM throughput, achieved occupancy, whether the bottleneck is memory or compute. It may be missing or blocked on Colab (`ERR_NVGPUCTRPERM`); if so, skip this cell.

In [ ]:
!which ncu && cd lab6 && ncu --kernel-name regex:"k_tiled|k_regblock" --launch-count 2 \
    --metrics sm__throughput.avg.pct_of_peak_sustained_elapsed,dram__throughput.avg.pct_of_peak_sustained_elapsed,sm__warps_active.avg.pct_of_peak_sustained_active \
    bin/sgemm regblock 2048 1 || echo "ncu not available here, skipping"

## 9. Calling CUDA from Python (handout Part 7)

Python cannot launch a CUDA kernel by itself, but it can call any C function in a **shared library** (`.so`) through `ctypes`. Three steps:

1. Write the C/CUDA functions inside `extern "C" { ... }`. Without it, the C++ compiler *mangles* names (e.g. `_Z19gpu_matrix_multiplyPKfS0_Pfi`) and Python cannot find them
2. Compile with `-shared -Xcompiler -fPIC` to get a `.so`
3. In Python: `ctypes.CDLL("./libmm.so")`, declare the argument types, pass NumPy arrays as pointers

The library below exposes two versions of the same multiply:

| Function | What a call does |
|---|---|
| `gpu_matrix_multiply` (handout interface) | `cudaMalloc` ×3 → copy A, B in → kernel → copy C out → `cudaFree` ×3, **every call** |
| `mm_persistent` | keeps the device buffers between calls, re-allocates only when N grows, and times the kernel alone with `cudaEvent` |

The comparison tells you **where the time goes**: allocation, transfers, or computation.

In [ ]:
%%writefile mm_lib.cu
#include <cuda_runtime.h>
#define TILE 16

__global__ void mm_tiled(const float *A, const float *B, float *C, int N)
{
    __shared__ float As[TILE][TILE], Bs[TILE][TILE];
    int tx = threadIdx.x, ty = threadIdx.y;
    int row = blockIdx.y * TILE + ty, col = blockIdx.x * TILE + tx;
    float acc = 0.0f;
    for (int m = 0; m < (N + TILE - 1) / TILE; ++m) {
        int aCol = m * TILE + tx, bRow = m * TILE + ty;
        As[ty][tx] = (row < N && aCol < N) ? A[row * N + aCol] : 0.0f;
        Bs[ty][tx] = (bRow < N && col < N) ? B[bRow * N + col] : 0.0f;
        __syncthreads();
        for (int k = 0; k < TILE; ++k) acc += As[ty][k] * Bs[k][tx];
        __syncthreads();
    }
    if (row < N && col < N) C[row * N + col] = acc;
}

static void launch(const float *dA, const float *dB, float *dC, int N)
{
    dim3 block(TILE, TILE), grid((N + TILE - 1) / TILE, (N + TILE - 1) / TILE);
    mm_tiled<<<grid, block>>>(dA, dB, dC, N);
}

// extern "C": keep plain C symbol names so ctypes can find them.
extern "C" {

// Handout interface: everything is allocated, copied and freed on every call.
// Returns 0 on success, otherwise the CUDA error code.
int gpu_matrix_multiply(const float *hA, const float *hB, float *hC, int N)
{
    size_t bytes = (size_t)N * N * sizeof(float);
    float *dA, *dB, *dC;
    cudaMalloc(&dA, bytes); cudaMalloc(&dB, bytes); cudaMalloc(&dC, bytes);
    cudaMemcpy(dA, hA, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(dB, hB, bytes, cudaMemcpyHostToDevice);
    launch(dA, dB, dC, N);
    cudaMemcpy(hC, dC, bytes, cudaMemcpyDeviceToHost);   // also waits for the kernel
    cudaFree(dA); cudaFree(dB); cudaFree(dC);
    return (int)cudaGetLastError();
}

// Persistent version: buffers survive between calls; the kernel is timed on its own.
static float *pA = nullptr, *pB = nullptr, *pC = nullptr;
static int capN = 0;
static float lastKernelMs = 0.0f;

int mm_persistent(const float *hA, const float *hB, float *hC, int N)
{
    size_t bytes = (size_t)N * N * sizeof(float);
    if (N > capN) {                                      // grow only when needed
        cudaFree(pA); cudaFree(pB); cudaFree(pC);
        cudaMalloc(&pA, bytes); cudaMalloc(&pB, bytes); cudaMalloc(&pC, bytes);
        capN = N;
    }
    cudaMemcpy(pA, hA, bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(pB, hB, bytes, cudaMemcpyHostToDevice);
    cudaEvent_t e0, e1;
    cudaEventCreate(&e0); cudaEventCreate(&e1);
    cudaEventRecord(e0);
    launch(pA, pB, pC, N);
    cudaEventRecord(e1);
    cudaEventSynchronize(e1);
    cudaEventElapsedTime(&lastKernelMs, e0, e1);
    cudaEventDestroy(e0); cudaEventDestroy(e1);
    cudaMemcpy(hC, pC, bytes, cudaMemcpyDeviceToHost);
    return (int)cudaGetLastError();
}

float mm_last_kernel_ms(void) { return lastKernelMs; }

}  // extern "C"

In [ ]:
!nvcc -O3 -arch=native -Xcompiler -fPIC -shared -o libmm.so mm_lib.cu
!nm -D libmm.so | grep -E "gpu_matrix_multiply|mm_persistent|mm_last"

`nm -D` lists the exported symbols: the names are plain, exactly as written, thanks to `extern "C"`. Try removing `extern "C"` and running `nm` again to see mangled names.

Now the Python side. `np.ctypeslib.ndpointer` tells ctypes "this argument is a contiguous float32 array", and ctypes passes the array's data pointer to C. Declaring `argtypes` matters: without it, ctypes guesses and a wrong guess is a crash, not an error message.

In [ ]:
import ctypes, time
import numpy as np

lib = ctypes.CDLL("./libmm.so")
F32 = np.ctypeslib.ndpointer(dtype=np.float32, ndim=1, flags="C_CONTIGUOUS")
for fn in (lib.gpu_matrix_multiply, lib.mm_persistent):
    fn.argtypes = [F32, F32, F32, ctypes.c_int]
    fn.restype = ctypes.c_int
lib.mm_last_kernel_ms.argtypes = []
lib.mm_last_kernel_ms.restype = ctypes.c_float

def gpu_matmul(A, B, persistent=False):
    """C = A @ B on the GPU. A and B are square float32 NumPy arrays."""
    N = A.shape[0]
    A = np.ascontiguousarray(A, dtype=np.float32)   # C code expects row-major float32
    B = np.ascontiguousarray(B, dtype=np.float32)
    C = np.empty((N, N), dtype=np.float32)
    fn = lib.mm_persistent if persistent else lib.gpu_matrix_multiply
    rc = fn(A.ravel(), B.ravel(), C.ravel(), N)     # ravel() of a contiguous array is a view: C is filled in place
    if rc != 0:
        raise RuntimeError(f"CUDA error {rc}")
    return C

# Correctness first
N = 1000                                            # not a multiple of 16: tests the boundary checks
A = np.random.rand(N, N).astype(np.float32)
B = np.random.rand(N, N).astype(np.float32)
C = gpu_matmul(A, B)
print("max |GPU - NumPy| =", np.abs(C - A @ B).max())

**Predict**: for N = 2048, which is larger: the kernel time or everything else in a `gpu_matrix_multiply` call?

In [ ]:
def best_ms(f, reps=5):
    f()                                             # warm-up (first call also creates the CUDA context)
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); f(); ts.append((time.perf_counter() - t0) * 1e3)
    return min(ts)

print(f"{'N':>5} {'NumPy (CPU)':>12} {'handout call':>13} {'persistent call':>16} {'kernel only':>12} {'kernel GFLOP/s':>15}")
for N in (256, 512, 1024, 2048, 4096):
    A = np.random.rand(N, N).astype(np.float32)
    B = np.random.rand(N, N).astype(np.float32)
    t_np = best_ms(lambda: A @ B, reps=3)
    t_hand = best_ms(lambda: gpu_matmul(A, B))
    t_pers = best_ms(lambda: gpu_matmul(A, B, persistent=True))
    k = lib.mm_last_kernel_ms()
    print(f"{N:5d} {t_np:10.2f}ms {t_hand:11.2f}ms {t_pers:14.2f}ms {k:10.2f}ms {2 * N**3 / (k * 1e-3) / 1e9:15.1f}")

**Reading the table**:
- **handout − persistent** = the cost of `cudaMalloc`/`cudaFree` on every call
- **persistent − kernel** = mostly PCIe transfers (3 × N² × 4 bytes) plus the Python/ctypes call overhead
- For small N, the GPU loses to NumPy: fixed overheads dominate. For large N, the N³ work dominates and the GPU wins
- NumPy calls an optimised multithreaded CPU BLAS: that is the fair CPU baseline, not the triple loop

> Gotcha: Python keeps a loaded `.so` in memory. If you edit `mm_lib.cu` and rebuild, **restart the runtime** (or build under a new file name), otherwise you are still calling the old code.

## 10. Image convolution (handout Part 8)

2-D convolution: each output pixel is a weighted sum of the N×N neighbourhood around it, with weights given by the filter.

```
out[y][x] = Σ_i Σ_j  img[y+i-r][x+j-r] · filt[i][j]        r = N/2, image edges clamped
```

Different filters do different things: average (blur), weight the centre (sharpen), take a difference (edge detection).

Two GPU ideas here:

1. **Constant memory for the filter**: all threads of a warp read *the same* filter tap at the same moment. Constant memory has a cache built for exactly that: one read, broadcast to all 32 threads
2. **Shared-memory tile with a halo**: a 16×16 block of outputs needs a (16+N−1)² window of input: the tile plus r pixels on every side (the *halo*). The block loads the window once, cooperatively, then each thread reads its N×N neighbourhood from shared memory instead of DRAM

```
┌─────────────────────┐
│ halo (r pixels)     │
│   ┌─────────────┐   │
│   │ 16×16 output│   │      shared window = (16 + N - 1)²
│   │    tile     │   │
│   └─────────────┘   │
│                     │
└─────────────────────┘
```

In [ ]:
%%writefile conv_lib.cu
#include <cuda_runtime.h>
#define MAX_N 31
#define TILE 16

__constant__ float c_filt[MAX_N * MAX_N];              // read-only, cached, broadcast to a warp

__device__ int clampi(int v, int lo, int hi) { return v < lo ? lo : (v > hi ? hi : v); }

// One thread per output pixel, every read from global memory.
__global__ void conv_naive(const unsigned char *img, float *out, int M, int N)
{
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;
    if (x >= M || y >= M) return;
    int r = N / 2;
    float acc = 0.0f;
    for (int i = 0; i < N; i++)
        for (int j = 0; j < N; j++)
            acc += img[clampi(y + i - r, 0, M - 1) * M + clampi(x + j - r, 0, M - 1)] * c_filt[i * N + j];
    out[y * M + x] = acc;
}

// The block first copies its (TILE + N - 1)^2 input window (tile + halo) into shared memory.
__global__ void conv_tiled(const unsigned char *img, float *out, int M, int N)
{
    extern __shared__ unsigned char win[];              // size chosen at launch time
    int r = N / 2, W = TILE + N - 1;
    int bx = blockIdx.x * TILE, by = blockIdx.y * TILE;
    // 256 threads cooperatively load W*W pixels: thread t loads t, t+256, t+512, ...
    for (int idx = threadIdx.y * TILE + threadIdx.x; idx < W * W; idx += TILE * TILE) {
        int wy = idx / W, wx = idx % W;
        win[idx] = img[clampi(by + wy - r, 0, M - 1) * M + clampi(bx + wx - r, 0, M - 1)];
    }
    __syncthreads();                                    // window complete before anyone reads it
    int x = bx + threadIdx.x, y = by + threadIdx.y;
    if (x >= M || y >= M) return;                       // after the barrier: everyone helped load
    float acc = 0.0f;
    for (int i = 0; i < N; i++)
        for (int j = 0; j < N; j++)
            acc += win[(threadIdx.y + i) * W + threadIdx.x + j] * c_filt[i * N + j];
    out[y * M + x] = acc;
}

extern "C" {

// Returns the kernel time in ms (or -1 on error). Allocates per call for simplicity.
float conv_gpu(const unsigned char *hImg, float *hOut, int M, const float *hFilt, int N, int tiled)
{
    if (N % 2 == 0 || N > MAX_N) return -1.0f;
    size_t px = (size_t)M * M;
    unsigned char *dImg; float *dOut;
    cudaMalloc(&dImg, px);
    cudaMalloc(&dOut, px * sizeof(float));
    cudaMemcpy(dImg, hImg, px, cudaMemcpyHostToDevice);
    cudaMemcpyToSymbol(c_filt, hFilt, N * N * sizeof(float));   // host -> constant memory

    dim3 block(TILE, TILE), grid((M + TILE - 1) / TILE, (M + TILE - 1) / TILE);
    cudaEvent_t e0, e1;
    cudaEventCreate(&e0); cudaEventCreate(&e1);
    cudaEventRecord(e0);
    if (tiled) conv_tiled<<<grid, block, (TILE + N - 1) * (TILE + N - 1)>>>(dImg, dOut, M, N);
    else       conv_naive<<<grid, block>>>(dImg, dOut, M, N);
    cudaEventRecord(e1);
    cudaEventSynchronize(e1);
    float ms;
    cudaEventElapsedTime(&ms, e0, e1);

    cudaMemcpy(hOut, dOut, px * sizeof(float), cudaMemcpyDeviceToHost);
    cudaFree(dImg); cudaFree(dOut);
    cudaEventDestroy(e0); cudaEventDestroy(e1);
    return cudaGetLastError() == cudaSuccess ? ms : -1.0f;
}

}  // extern "C"

In [ ]:
!nvcc -O3 -arch=native -Xcompiler -fPIC -shared -o libconv.so conv_lib.cu

Python side: the ctypes binding, a NumPy reference with the same semantics (correlation, clamp-to-edge borders), and the filters.

The NumPy reference uses a trick: pad the image by r pixels (`mode="edge"` = clamp), then for each filter tap (i, j) add a **shifted copy** of the whole image times that weight. N² vectorised array operations instead of M²·N² Python loop iterations.

In [ ]:
import ctypes, time
import numpy as np
import matplotlib.pyplot as plt

clib = ctypes.CDLL("./libconv.so")
U8 = np.ctypeslib.ndpointer(dtype=np.uint8, ndim=1, flags="C_CONTIGUOUS")
F32 = np.ctypeslib.ndpointer(dtype=np.float32, ndim=1, flags="C_CONTIGUOUS")
clib.conv_gpu.argtypes = [U8, F32, ctypes.c_int, F32, ctypes.c_int, ctypes.c_int]
clib.conv_gpu.restype = ctypes.c_float

def conv_gpu(img, filt, tiled=True):
    """img: square uint8 array, filt: odd square float array -> (float32 image, kernel ms)."""
    img = np.ascontiguousarray(img, dtype=np.uint8)
    filt = np.ascontiguousarray(filt, dtype=np.float32)
    out = np.empty(img.shape, dtype=np.float32)
    ms = clib.conv_gpu(img.ravel(), out.ravel(), img.shape[0], filt.ravel(), filt.shape[0], int(tiled))
    if ms < 0:
        raise RuntimeError("conv_gpu failed")
    return out, ms

def conv_ref(img, filt):
    """NumPy reference: pad with edge values, then add one shifted, weighted copy per filter tap."""
    n, M = filt.shape[0], img.shape[0]
    r = n // 2
    p = np.pad(img.astype(np.float64), r, mode="edge")
    out = np.zeros((M, M))
    for i in range(n):
        for j in range(n):
            out += filt[i, j] * p[i:i + M, j:j + M]
    return out.astype(np.float32)

g5 = np.array([1, 4, 6, 4, 1], np.float32)
FILTERS = {
    "box blur 3x3":  np.full((3, 3), 1 / 9, np.float32),         # average of the neighbourhood
    "gaussian 5x5":  np.outer(g5, g5) / 256,                      # smoother blur, centre weighted
    "box blur 15x15": np.full((15, 15), 1 / 225, np.float32),     # strong blur
    "sharpen":       np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]], np.float32),   # centre minus neighbours
    "sobel x":       np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], np.float32),    # horizontal gradient -> vertical edges
    "sobel y":       np.array([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], np.float32),    # vertical gradient -> horizontal edges
    "laplacian":     np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], np.float32),      # second derivative, all edges
    "emboss":        np.array([[-2, -1, 0], [-1, 1, 1], [0, 1, 2]], np.float32),
}

def test_image(M):
    try:
        from skimage import data
        from PIL import Image
        return np.asarray(Image.fromarray(data.camera()).resize((M, M)), np.uint8)
    except Exception:                                    # fallback: a synthetic pattern
        y, x = np.mgrid[0:M, 0:M] / M
        return np.clip(127 + 60 * np.sin(40 * x) * np.cos(25 * y) + 60 * ((x - .5)**2 + (y - .5)**2 < .08), 0, 255).astype(np.uint8)

img = test_image(1024)
print(img.shape, img.dtype)

Apply every filter on the GPU, check each against NumPy, and show the results. Edge filters produce negative values, so they are shown as the absolute value scaled to 0–255; the last panel combines both Sobel directions into the gradient magnitude √(gx² + gy²).

In [ ]:
def show(x, edge=False):
    if edge:
        x = np.abs(x)
        return x / max(x.max(), 1e-6) * 255
    return np.clip(x, 0, 255)

panels = {"original": img}
for name, f in FILTERS.items():
    out, ms = conv_gpu(img, f)
    err = np.abs(out - conv_ref(img, f)).max()
    print(f"{name:>15}: kernel {ms:6.3f} ms   max |GPU - NumPy| = {err:.1e}")
    panels[name] = show(out, edge=name.startswith(("sobel", "laplacian")))
gx, _ = conv_gpu(img, FILTERS["sobel x"])
gy, _ = conv_gpu(img, FILTERS["sobel y"])
panels["sobel magnitude"] = show(np.hypot(gx, gy), edge=True)

fig, axes = plt.subplots(2, 5, figsize=(18, 7.5))
for ax in axes.ravel():
    ax.axis("off")
for ax, (name, x) in zip(axes.ravel(), panels.items()):
    ax.imshow(x, cmap="gray", vmin=0, vmax=255)
    ax.set_title(name)
plt.tight_layout(); plt.show()

**Try it**: design your own filter, e.g. a 7×7 motion blur (`np.eye(7) / 7`, a diagonal line), or upload your own photo (Files panel on the left), load it with `PIL.Image.open(...).convert("L")` and resize it to a square.

### Performance: image size × filter size

**Predict**:
- How does the time grow with N? (Work per pixel is N².)
- Is the shared-memory tile worth it for N = 3? For N = 15? (Hint: for small N, the L1 cache already catches most of the reuse.)

In [ ]:
print(f"{'M':>5} {'N':>3} {'NumPy CPU':>10} {'GPU naive':>10} {'GPU tiled':>10} {'tiled call':>11} {'speed-up':>9}")
for M in (512, 2048, 4096):
    im = test_image(M)
    for N in (3, 7, 15):
        f = np.full((N, N), 1 / N**2, np.float32)
        conv_gpu(im, f)                                   # warm-up
        t0 = time.perf_counter(); conv_ref(im, f); t_np = (time.perf_counter() - t0) * 1e3
        _, k_naive = conv_gpu(im, f, tiled=False)
        t0 = time.perf_counter(); _, k_tiled = conv_gpu(im, f, tiled=True); t_call = (time.perf_counter() - t0) * 1e3
        print(f"{M:5d} {N:3d} {t_np:8.1f}ms {k_naive:8.3f}ms {k_tiled:8.3f}ms {t_call:9.2f}ms {t_np / t_call:8.0f}x")

**Reading the table**:
- `GPU naive` / `GPU tiled`: kernel time only. `tiled call`: the whole Python call, including allocation and PCIe copies. `speed-up`: NumPy time ÷ whole GPU call (the fair end-to-end comparison)
- The kernel is usually a small part of the call; the uint8 image goes in (M² bytes) and a float image comes back (4M² bytes)
- The tiled kernel's advantage grows with N: more reuse per loaded pixel

### The graded version

The repository runs the full Part 8 comparison: the plain C program (one core and OpenMP), the CUDA executable, and Python through the library with persistent buffers, saving every filtered image and a CSV.

In [ ]:
!cd lab6 && python3 python/conv_demo.py
from IPython.display import Image as Img, display
import glob
for g in sorted(glob.glob("lab6/results/conv/gallery_*.png")):
    display(Img(g))

## Summary: what each step taught

| Step | Idea | Bottleneck it removes |
|---|---|---|
| CPU ikj | contiguous access, SIMD | cache misses |
| naive GPU | thousands of threads | serial execution |
| coalescing | neighbouring threads → neighbouring addresses | wasted memory transactions |
| shared tiling | load once, reuse 16× | DRAM bandwidth |
| register blocking | 64 outputs per thread, outer product | shared-memory bandwidth |
| cuBLAS | years of hand tuning | everything above, done better |
| Tensor Cores | dedicated FP16 matrix hardware | FP32 arithmetic throughput (at a precision cost) |
| shared library | persistent buffers | malloc and PCIe on every call |
| convolution | constant memory + halo tile | redundant global reads |

The one sentence to remember: **GPU performance is mostly about moving data as little as possible, and in the right pattern.**